# Swing-base limit-bid — discovery

Inspired by the trader's BTC long limit at 75,485 on 2026-05-19 (spot ~76,500).
That level isn't arbitrary — it sits on the 2026-04-28..04-30 multi-day base
and at the top edge of a high-volume shelf in the spot volume profile.

**Hypothesis**: a discretionary 'limit-bid above the prior weekly swing low'
is a composite of edges that can be made systematic:

1. **Structure** — price is approaching a recently established N-day base
   (prior swing low + range expansion away from it).
2. **Profile** — bid sits at the top of an HVN shelf, above an LVN gap, above
   the stop cluster under the swing low.
3. **Forced shorting** — the print that made the swing low ran with negative
   spot/perp basis and negative funding (shorts paying into firm spot bid).
4. **Capitulation, not new shorts** — OI fell *during* the dump and rebuilt
   *after* the reclaim.
5. **Absorption** — spot CVD stayed positive across the wick-down hours.

This notebook builds those primitives and scores every historical 'approach
to a recent base' to ask: is the composite a real edge, or did it work once?

## Data

All from `data/databases/prod.db`:

- `cd_spot_15m`, `cd_futures_15m` — 15m OHLCV with buy/sell volume split (CVD)
- `cd_open_interest` — hourly OI
- `cd_funding_rate` — 8h funding
- `btc_1m` — minute bars, used for swing-base detection at finer resolution

In [ ]:
from __future__ import annotations

import datetime as dt
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('could not locate prod.db walking up from cwd')
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'databases' / 'prod.db'
print(f'DB: {DB}')

plt.style.use('dark_background')

## Load tables

In [ ]:
def _load(table: str, ts_col: str = 'timestamp', ts_unit: str = 's') -> pd.DataFrame:
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(f'SELECT * FROM {table} ORDER BY {ts_col}', con)
    con.close()
    df['ts'] = pd.to_datetime(df[ts_col], unit=ts_unit, utc=True)
    df = df.set_index('ts')
    return df[~df.index.duplicated(keep='last')]

spot15 = _load('cd_spot_15m')
fut15  = _load('cd_futures_15m')
oi_h   = _load('cd_open_interest')
fund_h = _load('cd_funding_rate')
# btc_1m uses open_time in ms
m1 = _load('btc_1m', ts_col='open_time', ts_unit='ms')

# Align everything to the OI start (earliest table that gates the composite).
start = oi_h.index.min()
spot15 = spot15.loc[start:]
fut15  = fut15.loc[start:]
fund_h = fund_h.loc[start:]
m1     = m1.loc[start:]
print(f'aligned start: {start}')
print(f'spot15 {len(spot15):,}  fut15 {len(fut15):,}  oi {len(oi_h):,}  funding {len(fund_h):,}  m1 {len(m1):,}')

## Build a 15m enriched frame

One row per 15m bar with:
- spot OHLCV + spot CVD delta
- futures OHLCV + perp CVD delta
- basis (futures_close − spot_close), in $ and bp
- OI close (forward-filled from hourly, limit 4 bars)
- funding (forward-filled from 8h, limit 32 bars)

In [ ]:
f = pd.DataFrame(index=spot15.index)
f['spot_o'] = spot15['open']; f['spot_h'] = spot15['high']
f['spot_l'] = spot15['low'];  f['spot_c'] = spot15['close']
f['spot_v'] = spot15['volume']
f['spot_cvd'] = spot15['volume_buy'] - spot15['volume_sell']

f['fut_o'] = fut15['open'].reindex(f.index)
f['fut_h'] = fut15['high'].reindex(f.index)
f['fut_l'] = fut15['low'].reindex(f.index)
f['fut_c'] = fut15['close'].reindex(f.index)
f['fut_v'] = fut15['volume'].reindex(f.index)
f['perp_cvd'] = (fut15['volume_buy'] - fut15['volume_sell']).reindex(f.index)

f['basis']    = f['fut_c'] - f['spot_c']
f['basis_bp'] = f['basis'] / f['spot_c'] * 10000.0

f['oi']      = oi_h['oi_close'].reindex(f.index).ffill(limit=4)
f['funding'] = fund_h['fr_close'].reindex(f.index).ffill(limit=32)

f = f.dropna(subset=['spot_c','fut_c','oi','funding']).copy()
print(f'{len(f):,} aligned 15m bars  ({f.index.min()} - {f.index.max()})')
f.tail(3)

## Swing-base detection

A **base** is a window of W hours where the lowest 15m close `B_low` was
set, price spent at least P% of those bars within `+R%` of `B_low`, then
price expanded **upward** by at least E% within the next K days. The base
becomes a *candidate level* once the upward expansion confirms.

Default parameters (tweak in the cell): W=36h, P≥40%, R=1.2%, E≥4%, K=3d.
These pick out the 2026-04-28..04-30 print as a base by 2026-05-02.

In [ ]:
BASE = {
    'window_hours':    36,    # W
    'cluster_pct':     0.40,  # P — fraction of W bars within R% of B_low
    'cluster_band_pct': 0.012, # R
    'expansion_pct':   0.04,  # E
    'expansion_days':  3,     # K
    'cooldown_hours':  24,    # do not emit overlapping bases
}

def detect_bases(df: pd.DataFrame, p=BASE) -> pd.DataFrame:
    """Return DataFrame indexed by base_confirm_ts with the level low."""
    bars_per_hr = 4
    W = p['window_hours'] * bars_per_hr
    K_bars = p['expansion_days'] * 24 * bars_per_hr
    cool_bars = p['cooldown_hours'] * bars_per_hr

    lo = df['spot_l'].values
    hi = df['spot_h'].values
    cl = df['spot_c'].values
    ts = df.index

    out = []
    i = W
    last_emit = -10**9
    while i + K_bars < len(df):
        win_lo = lo[i-W:i]
        b_low  = win_lo.min()
        # cluster: fraction of bars whose low <= b_low * (1+R)
        within = (win_lo <= b_low * (1 + p['cluster_band_pct'])).mean()
        if within >= p['cluster_pct']:
            # forward window
            fwd_hi = hi[i:i+K_bars].max()
            if fwd_hi >= b_low * (1 + p['expansion_pct']):
                # confirm when fwd_hi is first hit
                k_hit = i + int(np.argmax(hi[i:i+K_bars] >= b_low * (1 + p['expansion_pct'])))
                if k_hit - last_emit >= cool_bars:
                    out.append({
                        'base_start_ts': ts[i-W],
                        'base_end_ts':   ts[i-1],
                        'confirm_ts':    ts[k_hit],
                        'base_low':      float(b_low),
                        'cluster_frac':  float(within),
                        'fwd_high':      float(fwd_hi),
                    })
                    last_emit = k_hit
                    i = k_hit  # restart after confirmation to avoid heavy overlap
                    continue
        i += bars_per_hr  # step 1h
    return pd.DataFrame(out).set_index('confirm_ts')

bases = detect_bases(f)
print(f'{len(bases):,} historical bases detected')
bases.tail(8)

## Inspect the 2026-04-28..04-30 base

This is the base that motivated the 75,485 limit.  Confirm it survived the
detector and check its parameters.

In [ ]:
apr_bases = bases.loc['2026-04-15':'2026-05-10']
print('Bases confirmed in window:')
print(apr_bases[['base_low','cluster_frac','fwd_high']].round(2))

# nearest base whose low is closest to 75,485
if not apr_bases.empty:
    closest = (apr_bases['base_low'] - 75485).abs().idxmin()
    print(f'\nClosest base to 75,485 is confirmed at {closest}')
    print(apr_bases.loc[closest])

## Volume profile (visible range)

Bin spot volume by $250 over a rolling window and identify the value area
(70% of volume around the POC) plus the LVN/HVN structure under price.

In [ ]:
def volume_profile(df: pd.DataFrame, lookback_days: int = 30, bin_size: float = 250.0,
                   end_ts: pd.Timestamp | None = None) -> pd.DataFrame:
    end_ts = end_ts or df.index.max()
    start_ts = end_ts - pd.Timedelta(days=lookback_days)
    sl = df.loc[start_ts:end_ts]
    # use close price as bin key, weighted by volume
    bins = (sl['spot_c'] // bin_size * bin_size).astype(int)
    vp = sl.groupby(bins)['spot_v'].sum().rename('vol').to_frame()
    vp = vp.sort_index()
    vp['bin_lo'] = vp.index.astype(float)
    vp['bin_hi'] = vp.index.astype(float) + bin_size
    vp['pct'] = vp['vol'] / vp['vol'].sum()
    poc_bin = vp['vol'].idxmax()
    vp.attrs['poc'] = float(poc_bin) + bin_size / 2
    # value area: expand around POC until 70% volume
    target = vp['vol'].sum() * 0.70
    chosen = {poc_bin}
    cum = vp.loc[poc_bin, 'vol']
    bins_sorted = list(vp.index)
    pi = bins_sorted.index(poc_bin)
    lo_i, hi_i = pi, pi
    while cum < target and (lo_i > 0 or hi_i < len(bins_sorted)-1):
        v_lo = vp['vol'].iloc[lo_i-1] if lo_i > 0 else -1
        v_hi = vp['vol'].iloc[hi_i+1] if hi_i < len(bins_sorted)-1 else -1
        if v_lo >= v_hi and lo_i > 0:
            lo_i -= 1; cum += v_lo; chosen.add(bins_sorted[lo_i])
        elif hi_i < len(bins_sorted)-1:
            hi_i += 1; cum += v_hi; chosen.add(bins_sorted[hi_i])
        else:
            break
    vp.attrs['val'] = float(bins_sorted[lo_i])               # value-area low
    vp.attrs['vah'] = float(bins_sorted[hi_i]) + bin_size    # value-area high
    return vp

vp = volume_profile(f, lookback_days=35, end_ts=pd.Timestamp('2026-05-19', tz='UTC'))
print(f"POC={vp.attrs['poc']:.0f}  VAL={vp.attrs['val']:.0f}  VAH={vp.attrs['vah']:.0f}")

fig, ax = plt.subplots(figsize=(6.5, 7.5))
ax.barh(vp.index + 125, vp['vol'], height=240, color='#5588cc')
ax.axhline(vp.attrs['poc'], color='yellow', lw=1.0, label=f"POC {vp.attrs['poc']:.0f}")
ax.axhline(vp.attrs['val'], color='magenta', lw=0.8, label=f"VAL {vp.attrs['val']:.0f}")
ax.axhline(vp.attrs['vah'], color='magenta', lw=0.8, label=f"VAH {vp.attrs['vah']:.0f}")
ax.axhline(75485, color='lime', lw=1.2, ls='--', label='trader bid 75,485')
ax.set_title('Spot volume profile — last 35d ending 2026-05-19')
ax.set_ylabel('price ($)'); ax.set_xlabel('volume (BTC)')
ax.legend(loc='upper right', fontsize=8); plt.tight_layout(); plt.show()

## Confluence score at base confirmation

For every confirmed base, score the *base-forming window* on the four
non-structure legs:

| Leg | Test |
|---|---|
| **basis** | mean futures−spot basis during the base window ≤ −2 bp (futures discount) |
| **funding** | mean funding during the window < 0 (shorts paying) |
| **oi_flush** | OI fell from base_start to base_low bar by ≥ 1.5%, then recovered |
| **absorption** | spot CVD stayed net positive across the base window |

All four are *post-print signatures* — what the tape looked like while the
swing low was being set.  The structural leg is the base itself.

In [ ]:
SCORE = {
    'basis_bp_max':       -2.0,
    'oi_flush_pct':       0.015,
}

def score_base(row: pd.Series, df: pd.DataFrame, p=SCORE) -> pd.Series:
    sl = df.loc[row['base_start_ts']: row['base_end_ts']]
    if sl.empty:
        return pd.Series({'basis': 0, 'funding': 0, 'oi_flush': 0, 'absorption': 0,
                          'basis_bp_mean': np.nan, 'funding_mean': np.nan,
                          'oi_drawdown_pct': np.nan, 'spot_cvd_sum': np.nan})
    basis_mean = sl['basis_bp'].mean()
    fund_mean  = sl['funding'].mean()
    oi_start   = sl['oi'].iloc[0]
    oi_min     = sl['oi'].min()
    oi_dd      = (oi_min - oi_start) / max(oi_start, 1e-9)
    spot_cvd   = sl['spot_cvd'].sum()
    return pd.Series({
        'basis':         int(basis_mean <= p['basis_bp_max']),
        'funding':       int(fund_mean < 0),
        'oi_flush':      int(oi_dd <= -p['oi_flush_pct']),
        'absorption':    int(spot_cvd > 0),
        'basis_bp_mean': float(basis_mean),
        'funding_mean':  float(fund_mean),
        'oi_drawdown_pct': float(oi_dd),
        'spot_cvd_sum':  float(spot_cvd),
    })

scored = bases.join(bases.apply(score_base, df=f, axis=1))
scored['conf_score'] = scored[['basis','funding','oi_flush','absorption']].sum(axis=1)
print(f'{len(scored):,} bases scored')
print('score distribution:')
print(scored['conf_score'].value_counts().sort_index())
scored.tail(6)

## Multi-timeframe bias

The trader being studied stacks his bias across timeframes (e.g. *monthly
bullish, daily bearish, hourly bullish*). The confluence score above scores
*how* the swing low formed — not *whether the market structure says buy*.
Add a five-timeframe bias label to every base.

For each base's `confirm_ts`, look up the last *closed* bar on each TF and
label it:

| Symbol | Rule |
|---|---|
| `+` (bullish) | close > SMA **and** SMA rising over slope window |
| `−` (bearish) | close < SMA **and** SMA falling over slope window |
| `0` (neutral) | anything else (chop / inflecting) |

Defaults:

| TF | SMA period | Slope lookback |
|---|---|---|
| Monthly (M) | 12 | 3 |
| Weekly (W)  | 20 | 4 |
| Daily (D)   | 50 | 5 |
| 4-hour (H4) | 50 | 12 (2 days) |
| 1-hour (H1) | 50 | 24 (1 day) |

Output: 5-char signature like `++−++` (M+W+D−H4+H1+) per base.

We re-load `btc_1m` *un-trimmed* (back to 2017) so the monthly SMA is valid
for the earliest bases.

In [ ]:
# Reload btc_1m back to 2017 so monthly SMA is valid for the earliest bases.
con = sqlite3.connect(str(DB))
_m1_full = pd.read_sql('SELECT open_time, open, high, low, close, volume FROM btc_1m ORDER BY open_time', con)
con.close()
_m1_full['ts'] = pd.to_datetime(_m1_full['open_time'], unit='ms', utc=True)
_m1_full = _m1_full.set_index('ts').drop(columns=['open_time'])
_m1_full = _m1_full[~_m1_full.index.duplicated(keep='last')]

def _resample(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    return df.resample(rule).agg(o=('open','first'), h=('high','max'),
                                 l=('low','min'),   c=('close','last'),
                                 v=('volume','sum')).dropna()

H1 = _resample(_m1_full, '1h')
H4 = _resample(_m1_full, '4h')
D1 = _resample(_m1_full, '1D')
W1 = _resample(_m1_full, '1W')
M1 = _resample(_m1_full, '1ME')
print(f'H1 {len(H1):,} H4 {len(H4):,} D1 {len(D1):,} W1 {len(W1):,} M1 {len(M1):,}')

MTF = {
    'M':  {'tf': M1, 'sma': 12, 'slope': 3},
    'W':  {'tf': W1, 'sma': 20, 'slope': 4},
    'D':  {'tf': D1, 'sma': 50, 'slope': 5},
    'H4': {'tf': H4, 'sma': 50, 'slope': 12},
    'H1': {'tf': H1, 'sma': 50, 'slope': 24},
}

for k, cfg in MTF.items():
    tf = cfg['tf']
    tf['sma']      = tf['c'].rolling(cfg['sma']).mean()
    tf['sma_then'] = tf['sma'].shift(cfg['slope'])
    up = (tf['c'] > tf['sma']) & (tf['sma'] > tf['sma_then'])
    dn = (tf['c'] < tf['sma']) & (tf['sma'] < tf['sma_then'])
    tf['bias'] = np.where(up, '+', np.where(dn, '-', '0'))

def bias_at(ts: pd.Timestamp, label: str) -> str:
    tf = MTF[label]['tf']
    idx = tf.index.asof(ts)
    if pd.isna(idx):
        return '?'
    v = tf.loc[idx, 'bias']
    return v if isinstance(v, str) else '?'

for label in ['M','W','D','H4','H1']:
    scored[f'bias_{label}'] = scored.index.map(lambda ts, l=label: bias_at(ts, l))

scored['mtf_sig'] = scored.apply(
    lambda r: ''.join(r[f'bias_{k}'] for k in ['M','W','D','H4','H1']), axis=1)

# net direction: sum of '+' minus '-' (range -5..+5)
def _net(sig: str) -> int:
    return sig.count('+') - sig.count('-')
scored['mtf_net'] = scored['mtf_sig'].map(_net)

print('top 15 signatures (count):')
print(scored['mtf_sig'].value_counts().head(15).to_string())
print(f'\nmtf_net distribution:')
print(scored['mtf_net'].value_counts().sort_index().to_string())

## Forward-return analysis — position-trade with staged scale-out

The trader being studied holds positions for **weeks**, not days. We model
exits as a staged scale-out plus a trailing runner:

| Component | Rule |
|---|---|
| **Entry** | limit-bid at `base_low * (1 + entry_offset_pct)` |
| **Initial stop** | `base_low * (1 - stop_offset_pct)` — wider than swing scalp to survive multi-week chop |
| **Tier 1 (33%)** | close at `+R1` (default 3R) |
| **Tier 2 (33%)** | close at `+R2` (default 6R) |
| **Tier 3 (33%)** | trail at `trail_pct` (default 5%) under the running high-water mark, armed once `+R1` hits |
| **TIF** | hard close at `tif_days` (default 60) from `confirm_ts` |

Per-signal R is the position-weighted sum of each tier's outcome. The trail
is armed only after tier 1 fills (so a quick reversal still loses −1R on the
remaining 67% via the original SL).

In [ ]:
TRADE = {
    'entry_offset_pct':  0.007,   # bid +0.7% above base_low
    'stop_offset_pct':   0.020,   # stop -2.0% below base_low (~2.7% from entry)
    'target_R1':         3.0,     # tier 1 take-profit
    'target_R2':         6.0,     # tier 2 take-profit
    'trail_pct':         0.05,    # trail 5% under high-water mark, armed after tier 1
    'tier_size':         1/3,     # equal thirds
    'tif_days':          60,
}

def simulate(row: pd.Series, df: pd.DataFrame, p=TRADE) -> pd.Series:
    entry = row['base_low'] * (1 + p['entry_offset_pct'])
    stop_initial = row['base_low'] * (1 - p['stop_offset_pct'])
    risk = entry - stop_initial
    if risk <= 0:
        return pd.Series({'filled': 0, 't1': 0, 't2': 0, 'tr_hit': 0, 'sl_only': 0,
                          'r_outcome': np.nan, 'mfe_R': np.nan,
                          'hold_hours': np.nan, 'wait_to_fill_hours': np.nan})
    t1_px = entry + p['target_R1'] * risk
    t2_px = entry + p['target_R2'] * risk
    tif_end = row.name + pd.Timedelta(days=p['tif_days'])
    sl = df.loc[row.name:tif_end]
    if sl.empty:
        return pd.Series({'filled': 0, 't1': 0, 't2': 0, 'tr_hit': 0, 'sl_only': 0,
                          'r_outcome': np.nan, 'mfe_R': np.nan,
                          'hold_hours': np.nan, 'wait_to_fill_hours': np.nan})
    fill_mask = sl['spot_l'] <= entry
    if not fill_mask.any():
        return pd.Series({'filled': 0, 't1': 0, 't2': 0, 'tr_hit': 0, 'sl_only': 0,
                          'r_outcome': 0.0, 'mfe_R': 0.0,
                          'hold_hours': np.nan, 'wait_to_fill_hours': np.nan})
    fill_ts = sl.index[fill_mask.argmax()]
    wait_hours = (fill_ts - row.name).total_seconds() / 3600
    after = sl.loc[fill_ts:]

    # State machine: tier 1, tier 2, runner with trailing stop
    remaining = 1.0
    realized_R = 0.0
    t1_done = False
    t2_done = False
    trail_armed = False
    high_water = entry
    active_stop = stop_initial
    exit_ts = None

    for ts_, bar in after.iterrows():
        bar_h, bar_l, bar_c = float(bar['spot_h']), float(bar['spot_l']), float(bar['spot_c'])
        # Update high-water mark on the way up
        high_water = max(high_water, bar_h)
        # If trail is armed, raise active_stop
        if trail_armed:
            new_stop = high_water * (1 - p['trail_pct'])
            if new_stop > active_stop:
                active_stop = new_stop
        # Conservative order: stop first, then tier 1, then tier 2 within the same bar
        if bar_l <= active_stop:
            # close whatever is left at active_stop
            r_at_stop = (active_stop - entry) / risk
            realized_R += remaining * r_at_stop
            remaining = 0.0
            exit_ts = ts_
            break
        # Tier 1
        if not t1_done and bar_h >= t1_px:
            realized_R += p['tier_size'] * p['target_R1']
            remaining -= p['tier_size']
            t1_done = True
            trail_armed = True
            new_stop = high_water * (1 - p['trail_pct'])
            if new_stop > active_stop:
                active_stop = new_stop
        # Tier 2 (only if tier 1 hit and price kept going)
        if t1_done and not t2_done and bar_h >= t2_px:
            realized_R += p['tier_size'] * p['target_R2']
            remaining -= p['tier_size']
            t2_done = True
        # If remaining == 0 the runner is done (shouldn't happen here, but guard)
        if remaining <= 1e-9:
            exit_ts = ts_
            break

    if exit_ts is None:
        # TIF expiry: mark remaining to last close
        last_c = float(after['spot_c'].iloc[-1])
        r_last = (last_c - entry) / risk
        realized_R += remaining * r_last
        exit_ts = after.index[-1]

    mfe_R = float((after['spot_h'].max() - entry) / risk)
    hold_hours = (exit_ts - fill_ts).total_seconds() / 3600
    sl_only = int((not t1_done) and remaining == 0.0)
    return pd.Series({
        'filled':    1,
        't1':        int(t1_done),
        't2':        int(t2_done),
        'tr_hit':    int(t1_done and remaining == 0.0),  # runner closed on trail or stop after t1
        'sl_only':   sl_only,
        'r_outcome': float(realized_R),
        'mfe_R':     mfe_R,
        'hold_hours':       hold_hours,
        'wait_to_fill_hours': wait_hours,
    })

outcomes = scored.apply(simulate, df=f, axis=1)
results = scored.join(outcomes)
filled = results[results['filled'] == 1]
print(f'fill rate: {results["filled"].mean():.2%}   '
      f't1 rate (filled): {filled["t1"].mean():.2%}   '
      f't2 rate (filled): {filled["t2"].mean():.2%}   '
      f'sl_only (filled): {filled["sl_only"].mean():.2%}')
print(f'expectancy per signal: {results["r_outcome"].mean():.2f} R   '
      f'(per fill: {filled["r_outcome"].mean():.2f} R)')
print(f'median hold (filled): {filled["hold_hours"].median()/24:.1f} days   '
      f'90th pct: {filled["hold_hours"].quantile(0.9)/24:.1f} days')

## Stratify by confluence score

In [ ]:
by_score = results.groupby('conf_score').agg(
    n=('r_outcome', 'size'),
    fill_rate=('filled', 'mean'),
    t1_rate=('t1', 'mean'),
    t2_rate=('t2', 'mean'),
    sl_only=('sl_only', 'mean'),
    mean_R=('r_outcome', 'mean'),
    median_R=('r_outcome', 'median'),
    mfe_R=('mfe_R', 'mean'),
    hold_d_med=('hold_hours', lambda s: s.median()/24 if s.notna().any() else np.nan),
).round(3)
print('expectancy by confluence score (0..4)')
print(by_score)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(by_score.index, by_score['mean_R'], color='cyan', alpha=0.7)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('confluence score (0–4)'); ax.set_ylabel('mean R per signal')
ax.set_title('Swing-base limit-bid expectancy (staged scale-out)')
for x, v, n in zip(by_score.index, by_score['mean_R'], by_score['n']):
    ax.text(x, v, f'  n={n}', va='bottom' if v >= 0 else 'top', fontsize=8)
plt.tight_layout(); plt.show()

## Expectancy by MTF alignment signature

Test the trader's stated framework: does buying a base only when *higher TFs*
are bullish (regardless of lower-TF dip direction) lift expectancy?

Two views:
1. **By net** — bullish minus bearish TF count (range −5..+5). One number per base.
2. **By exact signature** — only signatures with ≥ 8 historical samples to avoid
   curve-fitting on tiny groups.

In [ ]:
# Re-join bias columns onto results (results was created before MTF cell ran)
for col in ['bias_M','bias_W','bias_D','bias_H4','bias_H1','mtf_sig','mtf_net']:
    results[col] = scored[col]

if 'year' not in results.columns:
    results['year'] = pd.to_datetime(results.index).year

print('=== Expectancy by mtf_net (bullish minus bearish TF count) ===')
by_net = results.groupby('mtf_net').agg(
    n=('r_outcome','size'),
    fill_rate=('filled','mean'),
    t1_rate=('t1','mean'),
    t2_rate=('t2','mean'),
    mean_R=('r_outcome','mean'),
    median_R=('r_outcome','median'),
).round(3)
print(by_net)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(by_net.index, by_net['mean_R'], color='cyan', alpha=0.7)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('mtf_net (-5 = all TFs bearish, +5 = all bullish)')
ax.set_ylabel('mean R per signal')
ax.set_title('Swing-base limit-bid expectancy by MTF net bias')
for x, v, n in zip(by_net.index, by_net['mean_R'], by_net['n']):
    ax.text(x, v, f'  n={n}', va='bottom' if v >= 0 else 'top', fontsize=8)
plt.tight_layout(); plt.show()

print('\n=== Expectancy by exact signature (n >= 8) ===')
by_sig = results.groupby('mtf_sig').agg(
    n=('r_outcome','size'),
    fill_rate=('filled','mean'),
    t1_rate=('t1','mean'),
    mean_R=('r_outcome','mean'),
).round(3)
by_sig = by_sig[by_sig['n'] >= 8].sort_values('mean_R', ascending=False)
print(by_sig.to_string())

print('\n=== Cross-tab: mtf_net x conf_score (mean_R, only cells with n>=5) ===')
cross = results.groupby(['mtf_net','conf_score']).agg(
    n=('r_outcome','size'), mean_R=('r_outcome','mean')
).round(2)
cross = cross[cross['n'] >= 5]
print(cross.to_string())

print('\n=== Year x mtf_net mean_R ===')
yr_net = results.groupby(['year','mtf_net'])['r_outcome'].agg(['size','mean']).round(3)
yr_net = yr_net[yr_net['size'] >= 3]
print(yr_net.to_string())

## Year breakdown — is the edge era-stable?

In [ ]:
results = results.copy()
results['year'] = pd.to_datetime(results.index).year
yr = results.groupby('year').agg(
    n=('r_outcome', 'size'),
    mean_R=('r_outcome', 'mean'),
    fill_rate=('filled', 'mean'),
    t1_rate=('t1', 'mean'),
    t2_rate=('t2', 'mean'),
    sl_only=('sl_only', 'mean'),
).round(3)
print(yr)

# high-score only
hi = results[results['conf_score'] >= 3]
yr_hi = hi.groupby('year').agg(
    n=('r_outcome', 'size'),
    mean_R=('r_outcome', 'mean'),
    fill_rate=('filled', 'mean'),
    t1_rate=('t1', 'mean'),
    t2_rate=('t2', 'mean'),
).round(3)
print(f'\n=== high confluence (score >= 3) only ===')
print(yr_hi)

## Case study — the 2026-04-28..04-30 base + the present 75,485 setup

Render the base, the bid level, and the structure around it.

In [ ]:
def plot_base(base_row: pd.Series, df: pd.DataFrame, extra_bid: float | None = None,
              forward_days: int = 60):
    start = base_row['base_start_ts'] - pd.Timedelta(hours=12)
    end   = base_row.name + pd.Timedelta(days=forward_days)
    sl = df.loc[start:end]

    fig, axes = plt.subplots(
        4, 1, figsize=(11, 9.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1.2, 1.2, 1.5]}
    )
    axes[0].plot(sl.index, sl['spot_c'], color='white', lw=1.0, label='spot')
    axes[0].plot(sl.index, sl['fut_c'],  color='orange', lw=0.8, alpha=0.7, label='perp')
    axes[0].axhline(base_row['base_low'], color='magenta', lw=1.0, ls='--',
                    label=f"base low {base_row['base_low']:.0f}")
    entry  = base_row['base_low'] * (1 + TRADE['entry_offset_pct'])
    stop_i = base_row['base_low'] * (1 - TRADE['stop_offset_pct'])
    risk   = entry - stop_i
    t1_px  = entry + TRADE['target_R1'] * risk
    t2_px  = entry + TRADE['target_R2'] * risk
    axes[0].axhline(entry,  color='lime', lw=0.8, ls=':', label=f'limit-bid {entry:.0f}')
    axes[0].axhline(stop_i, color='red',  lw=0.8, ls=':', label=f'initial stop {stop_i:.0f}')
    axes[0].axhline(t1_px,  color='cyan', lw=0.8, ls=':', label=f't1 +{TRADE["target_R1"]:.0f}R {t1_px:.0f}')
    axes[0].axhline(t2_px,  color='deepskyblue', lw=0.8, ls=':', label=f't2 +{TRADE["target_R2"]:.0f}R {t2_px:.0f}')
    if extra_bid is not None:
        axes[0].axhline(extra_bid, color='yellow', lw=1.2, label=f'trader bid {extra_bid:.0f}')
    axes[0].axvspan(base_row['base_start_ts'], base_row['base_end_ts'],
                    color='magenta', alpha=0.08, label='base window')
    axes[0].axvline(base_row.name, color='cyan', lw=0.8, ls='--', alpha=0.6, label='confirm')
    axes[0].set_title(f"base confirmed {base_row.name.strftime('%Y-%m-%d %H:%M')} | "
                      f"score {int(base_row.get('conf_score', -1))}/4")
    axes[0].set_ylabel('price'); axes[0].legend(loc='upper left', fontsize=7, ncol=2)

    axes[1].plot(sl.index, sl['basis_bp'], color='cyan', lw=0.9)
    axes[1].axhline(0, color='gray', lw=0.5)
    axes[1].axhline(SCORE['basis_bp_max'], color='magenta', lw=0.4, ls=':')
    axes[1].set_ylabel('basis (bp)')

    axes[2].plot(sl.index, sl['oi'] / 1000.0, color='violet', lw=0.9)
    axes[2].set_ylabel('OI (k BTC)')

    axes[3].plot(sl.index, sl['spot_cvd'].cumsum(), color='lime', lw=0.9, label='spot CVD')
    axes[3].plot(sl.index, sl['perp_cvd'].cumsum(), color='red',  lw=0.9, label='perp CVD')
    axes[3].axhline(0, color='gray', lw=0.5)
    axes[3].set_ylabel('cum CVD'); axes[3].legend(loc='upper left', fontsize=7)

    plt.tight_layout(); plt.show()

if not apr_bases.empty:
    target_idx = (apr_bases['base_low'] - 75485).abs().idxmin()
    row = results.loc[target_idx]
    print(row[['base_low','cluster_frac','basis_bp_mean','funding_mean',
               'oi_drawdown_pct','spot_cvd_sum','conf_score',
               'filled','t1','t2','sl_only','r_outcome','hold_hours']])
    plot_base(row, f, extra_bid=75485.0)

## Top historical hits (score == 4)

In [ ]:
perfect = results[results['conf_score'] == 4].sort_values('r_outcome', ascending=False)
print(f'{len(perfect)} bases with full 4/4 confluence')
print(perfect[['base_low','basis_bp_mean','funding_mean','oi_drawdown_pct',
               'spot_cvd_sum','filled','t1','t2','sl_only','r_outcome','hold_hours']].round(3).to_string())

for idx in perfect.index[:3]:
    plot_base(results.loc[idx], f)

## Is the current setup (2026-05-19, 75,485) a match?

Check whether the most recent base would be flagged today and what the
scored composite looks like *as of now* — useful as a live-replication smoke test.

In [ ]:
now = f.index.max()
recent = results.loc[results.index <= now].tail(8)
print('Most recent 8 scored bases (with MTF signature):')
cols = ['base_low','conf_score','mtf_sig','mtf_net',
        'basis_bp_mean','funding_mean','oi_drawdown_pct']
print(recent[cols].round(4).to_string())

# Where would the trader's 75,485 limit sit vs detector?
match_idx = (recent['base_low'] - 75485).abs().idxmin()
match = recent.loc[match_idx]
print(f'\nNearest base to 75,485 is at {match_idx}')
print(f'  base_low        = {match["base_low"]:.0f}  '
      f'(trader_bid - base_low = {75485 - match["base_low"]:+.0f})')
print(f'  detector entry  = {match["base_low"] * (1 + TRADE["entry_offset_pct"]):.0f}')
print(f'  conf_score      = {int(match["conf_score"])}/4')
print(f'  MTF signature   = {match["mtf_sig"]}  (M-W-D-H4-H1)')
print(f'  mtf_net         = {int(match["mtf_net"]):+d}')

# Cross-reference with the best-cell table we computed earlier
cell_n  = int(match['mtf_net'])
cell_cs = int(match['conf_score'])
subset = results[(results['mtf_net']==cell_n) & (results['conf_score']==cell_cs)]
if len(subset) >= 3:
    print(f'\nHistorical cell (mtf_net={cell_n:+d}, conf_score={cell_cs}): '
          f'n={len(subset)}, mean_R={subset["r_outcome"].mean():+.2f}, '
          f't1_rate={subset["t1"].mean():.2%}, t2_rate={subset["t2"].mean():.2%}')
else:
    print(f'\nHistorical cell (mtf_net={cell_n:+d}, conf_score={cell_cs}) has only n={len(subset)} — too few to project.')

# Also show what the 2026-04-22 analog's signature was (the +1.92R historical case)
april_match = results.loc['2026-04-22 05:15:00+00:00']
print(f'\n2026-04-22 analog (the +1.92R case):')
print(f'  base_low={april_match["base_low"]:.0f}  conf_score={int(april_match["conf_score"])}/4  '
      f'sig={april_match["mtf_sig"]}  mtf_net={int(april_match["mtf_net"]):+d}  '
      f'r_outcome={april_match["r_outcome"]:+.2f}R')

## Symmetric — short side

The trader takes both directions. So we mirror the entire pipeline:

| Long | Short |
|---|---|
| swing-low base (low cluster) | swing-high range (high cluster) |
| limit-bid at `base_low * (1 + offset)` | limit-sell at `top_high * (1 - offset)` |
| stop below `base_low` | stop above `top_high` |
| basis ≤ −2 bp (futures discount) | basis ≥ +2 bp (futures premium) |
| funding < 0 (shorts paying) | funding > 0 (longs paying) |
| OI flushed during dump | OI grew into the high (longs piled in) |
| spot CVD > 0 (absorption) | spot CVD < 0 (distribution at the high) |

MTF bias interpretation flips: for shorts the most favorable signature is
`++---` (HTF bullish, lower-TF rolling over — selling a top in a bull) or
`-----` (continuation short in trend down).

In [ ]:
def detect_tops(df: pd.DataFrame, p=BASE) -> pd.DataFrame:
    bph = 4
    W = p['window_hours'] * bph
    K_bars = p['expansion_days'] * 24 * bph
    cool_bars = p['cooldown_hours'] * bph
    hi = df['spot_h'].values; lo = df['spot_l'].values; ts = df.index
    out = []; i = W; last_emit = -10**9
    while i + K_bars < len(df):
        win_hi = hi[i-W:i]
        t_high = win_hi.max()
        within = (win_hi >= t_high * (1 - p['cluster_band_pct'])).mean()
        if within >= p['cluster_pct']:
            fwd_lo = lo[i:i+K_bars].min()
            if fwd_lo <= t_high * (1 - p['expansion_pct']):
                k_hit = i + int(np.argmax(lo[i:i+K_bars] <= t_high * (1 - p['expansion_pct'])))
                if k_hit - last_emit >= cool_bars:
                    out.append({'top_start_ts': ts[i-W], 'top_end_ts': ts[i-1],
                                'confirm_ts': ts[k_hit], 'top_high': float(t_high),
                                'cluster_frac': float(within), 'fwd_low': float(fwd_lo)})
                    last_emit = k_hit; i = k_hit; continue
        i += bph
    return pd.DataFrame(out).set_index('confirm_ts')

tops = detect_tops(f)
print(f'{len(tops):,} historical tops detected')

def score_top(row: pd.Series, df: pd.DataFrame, p=SCORE) -> pd.Series:
    sl = df.loc[row['top_start_ts']:row['top_end_ts']]
    if sl.empty:
        return pd.Series({'basis':0,'funding':0,'oi_pump':0,'distribution':0,
                          'basis_bp_mean':np.nan,'funding_mean':np.nan,
                          'oi_growth_pct':np.nan,'spot_cvd_sum':np.nan})
    basis_mean = sl['basis_bp'].mean(); fund_mean = sl['funding'].mean()
    oi_start = sl['oi'].iloc[0]; oi_max = sl['oi'].max()
    oi_growth = (oi_max - oi_start) / max(oi_start, 1e-9)
    spot_cvd = sl['spot_cvd'].sum()
    return pd.Series({
        'basis':         int(basis_mean >= -p['basis_bp_max']),   # mirror: ≥ +2 bp
        'funding':       int(fund_mean > 0),
        'oi_pump':       int(oi_growth >= p['oi_flush_pct']),
        'distribution':  int(spot_cvd < 0),
        'basis_bp_mean': float(basis_mean),
        'funding_mean':  float(fund_mean),
        'oi_growth_pct': float(oi_growth),
        'spot_cvd_sum':  float(spot_cvd),
    })

scored_s = tops.join(tops.apply(score_top, df=f, axis=1))
scored_s['conf_score'] = scored_s[['basis','funding','oi_pump','distribution']].sum(axis=1)

# attach MTF bias columns (re-use bias_at from the long-side cells)
for lbl in ['M','W','D','H4','H1']:
    scored_s[f'bias_{lbl}'] = scored_s.index.map(lambda ts, l=lbl: bias_at(ts, l))
scored_s['mtf_sig'] = scored_s.apply(
    lambda r: ''.join(r[f'bias_{k}'] for k in ['M','W','D','H4','H1']), axis=1)
scored_s['mtf_net'] = scored_s['mtf_sig'].map(lambda s: s.count('+') - s.count('-'))

print(f'{len(scored_s):,} tops scored')
print('short score distribution:')
print(scored_s['conf_score'].value_counts().sort_index().to_string())

def simulate_short(row: pd.Series, df: pd.DataFrame, p=TRADE) -> pd.Series:
    entry = row['top_high'] * (1 - p['entry_offset_pct'])
    stop_initial = row['top_high'] * (1 + p['stop_offset_pct'])
    risk = stop_initial - entry
    if risk <= 0:
        return pd.Series({'filled':0,'t1':0,'t2':0,'sl_only':0,'r_outcome':np.nan,
                          'mfe_R':np.nan,'hold_hours':np.nan,'wait_to_fill_hours':np.nan})
    t1_px = entry - p['target_R1'] * risk
    t2_px = entry - p['target_R2'] * risk
    tif_end = row.name + pd.Timedelta(days=p['tif_days'])
    sl = df.loc[row.name:tif_end]
    if sl.empty:
        return pd.Series({'filled':0,'t1':0,'t2':0,'sl_only':0,'r_outcome':np.nan,
                          'mfe_R':np.nan,'hold_hours':np.nan,'wait_to_fill_hours':np.nan})
    fill_mask = sl['spot_h'] >= entry
    if not fill_mask.any():
        return pd.Series({'filled':0,'t1':0,'t2':0,'sl_only':0,'r_outcome':0.0,'mfe_R':0.0,
                          'hold_hours':np.nan,'wait_to_fill_hours':np.nan})
    fill_ts = sl.index[fill_mask.argmax()]
    wait_hours = (fill_ts - row.name).total_seconds() / 3600
    after = sl.loc[fill_ts:]
    remaining = 1.0; realized_R = 0.0
    t1_done = False; t2_done = False; trail_armed = False
    low_water = entry; active_stop = stop_initial; exit_ts = None
    for ts_, bar in after.iterrows():
        bh, bl, bc = float(bar['spot_h']), float(bar['spot_l']), float(bar['spot_c'])
        low_water = min(low_water, bl)
        if trail_armed:
            new_stop = low_water * (1 + p['trail_pct'])
            if new_stop < active_stop: active_stop = new_stop
        if bh >= active_stop:
            r_at_stop = (entry - active_stop) / risk
            realized_R += remaining * r_at_stop
            remaining = 0.0; exit_ts = ts_; break
        if not t1_done and bl <= t1_px:
            realized_R += p['tier_size'] * p['target_R1']
            remaining -= p['tier_size']; t1_done = True; trail_armed = True
            new_stop = low_water * (1 + p['trail_pct'])
            if new_stop < active_stop: active_stop = new_stop
        if t1_done and not t2_done and bl <= t2_px:
            realized_R += p['tier_size'] * p['target_R2']
            remaining -= p['tier_size']; t2_done = True
        if remaining <= 1e-9:
            exit_ts = ts_; break
    if exit_ts is None:
        last_c = float(after['spot_c'].iloc[-1])
        realized_R += remaining * ((entry - last_c) / risk)
        exit_ts = after.index[-1]
    mfe_R = float((entry - after['spot_l'].min()) / risk)
    hold_hours = (exit_ts - fill_ts).total_seconds() / 3600
    sl_only = int((not t1_done) and remaining == 0.0)
    return pd.Series({'filled':1,'t1':int(t1_done),'t2':int(t2_done),'sl_only':sl_only,
                      'r_outcome':float(realized_R),'mfe_R':mfe_R,
                      'hold_hours':hold_hours,'wait_to_fill_hours':wait_hours})

short_out = scored_s.apply(simulate_short, df=f, axis=1)
results_s = scored_s.join(short_out)
results_s['year'] = pd.to_datetime(results_s.index).year
filled_s = results_s[results_s['filled']==1]
print(f'\nshorts headline: fill={results_s["filled"].mean():.2%} '
      f't1={filled_s["t1"].mean():.2%} t2={filled_s["t2"].mean():.2%} '
      f'sl_only={filled_s["sl_only"].mean():.2%}')
print(f'mean R per signal: {results_s["r_outcome"].mean():+.2f}   '
      f'per fill: {filled_s["r_outcome"].mean():+.2f}')
print(f'median hold (filled): {filled_s["hold_hours"].median()/24:.1f}d  '
      f'90th pct: {filled_s["hold_hours"].quantile(0.9)/24:.1f}d')

## Short-side expectancy stratified

By confluence score, MTF net, and year — mirror of the long-side analysis.

In [ ]:
print('=== short: expectancy by conf_score ===')
by_score_s = results_s.groupby('conf_score').agg(
    n=('r_outcome','size'), fill_rate=('filled','mean'),
    t1_rate=('t1','mean'), t2_rate=('t2','mean'),
    mean_R=('r_outcome','mean'), median_R=('r_outcome','median'),
).round(3)
print(by_score_s.to_string())

print('\n=== short: expectancy by mtf_net ===')
by_net_s = results_s.groupby('mtf_net').agg(
    n=('r_outcome','size'), fill_rate=('filled','mean'),
    t1_rate=('t1','mean'), mean_R=('r_outcome','mean'),
).round(3)
print(by_net_s.to_string())

print('\n=== short: cross-tab mtf_net x conf_score (n>=5) ===')
cross_s = results_s.groupby(['mtf_net','conf_score']).agg(
    n=('r_outcome','size'), mean_R=('r_outcome','mean')
).round(2)
cross_s = cross_s[cross_s['n'] >= 5]
print(cross_s.to_string())

print('\n=== short: top exact signatures (n>=6) ===')
by_sig_s = results_s.groupby('mtf_sig').agg(
    n=('r_outcome','size'), t1_rate=('t1','mean'), mean_R=('r_outcome','mean'),
).round(3)
print(by_sig_s[by_sig_s['n']>=6].sort_values('mean_R', ascending=False).to_string())

print('\n=== short: year breakdown ===')
yr_s = results_s.groupby('year').agg(
    n=('r_outcome','size'), mean_R=('r_outcome','mean'),
    fill_rate=('filled','mean'), t1_rate=('t1','mean'),
).round(3)
print(yr_s.to_string())

# Combined long+short universe — total trade frequency
print(f'\n=== Combined universe ===')
n_long  = len(results);  n_short = len(results_s)
n_hi_l = (results['conf_score']>=3).sum()
n_hi_s = (results_s['conf_score']>=3).sum()
days = (f.index.max() - f.index.min()).days
print(f'  long signals:  {n_long}  ({n_hi_l} at conf>=3)   over {days} days')
print(f'  short signals: {n_short}  ({n_hi_s} at conf>=3)')
print(f'  combined:      {n_long+n_short}  ({n_hi_l+n_hi_s} at conf>=3)')
print(f'  signal freq:   {(n_long+n_short)/days*7:.1f}/week '
      f'(any score)   {(n_hi_l+n_hi_s)/days*7:.2f}/week (conf>=3)')

# Implied weekly R (mixing long & short)
long_R  = results['r_outcome'].sum()
short_R = results_s['r_outcome'].sum()
print(f'  total realized R: long={long_R:+.1f}  short={short_R:+.1f}  combined={long_R+short_R:+.1f}')
print(f'  per week: {(long_R+short_R)/days*7:+.2f}R/week  '
      f'(if 1R = 1% of account: ~{(long_R+short_R)/days*7:.2f}%/week)')

## Next steps

- **Parameter sweep** — vary `window_hours`, `cluster_pct`, `entry_offset_pct`,
  `target_R`. Plot expectancy contours; pick robust ridge, not the peak.
- **Volume-profile leg** — promote the LVN→HVN test into the score (bid must
  sit at the top of an HVN ≥ 1.5× the bin below it).
- **Conservative SL/TP attribution** — current sim assumes SL first when both
  are hit on the same bar; cross-check on 1m bars from `btc_1m` for bases
  where this changes the outcome.
- **Walk-forward** — train threshold on rolling 18-month window, test on next
  6 months; report out-of-sample expectancy.
- **ETH version** — apply the same detector to `eth_1m` + ETH spot/futures
  tables; see whether the edge generalises.
- **Strategy wiring** — if expectancy holds OOS, draft a `swing_base_limit_bid`
  sleeve under [`strategies/`](../../../strategies/) emitting limit orders
  through the trade emitter, sized via `services/margin_sim.py`.